# 🛍️ Shopping Dataset — Python & Pandas Data Exploration Project

**Objective:** Learn Python basics and perform data exploration and cleaning using Pandas.

**Dataset:** Shopping Dataset (`shopping_dataset.csv`)

**Steps covered:**
1. Load a CSV dataset into a Pandas DataFrame  
2. Explore data (head/tail, shape, columns, data types)  
3. Handle missing values (identify, fill/drop)  
4. Perform basic operations (filter rows, select columns)  
5. Remove duplicates  
6. Create a derived column (`total_amount = price × quantity`)  
7. Save the cleaned dataset as a new CSV file  

## Step 0 — Import Libraries

In [ ]:
# Standard imports for data analysis
import pandas as pd
import numpy as np

# Display settings — show all columns, 2 decimal places for floats
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Libraries imported successfully')

## Step 1 — Load the CSV Dataset

We use `pd.read_csv()` to load our shopping data into a **DataFrame** —  
Pandas' core data structure, similar to a spreadsheet with rows and columns.

In [ ]:
# Load the dataset
df = pd.read_csv('shopping_dataset.csv')

print(f'✅ Dataset loaded successfully!')
print(f'   Shape: {df.shape[0]} rows × {df.shape[1]} columns')

## Step 2 — Explore the Data

### 2a. First 5 rows — `df.head()`

In [ ]:
df.head()

### 2b. Last 5 rows — `df.tail()`

In [ ]:
df.tail()

### 2c. Shape — (rows, columns)

In [ ]:
rows, cols = df.shape
print(f'Rows   : {rows}')
print(f'Columns: {cols}')

### 2d. Column names

In [ ]:
print('Columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i}. {col}')

### 2e. Data types of each column

In [ ]:
print(df.dtypes)
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

### 2f. Statistical summary — `df.describe()`

In [ ]:
df.describe()

### 2g. Value counts — Category & Gender distribution

In [ ]:
print('Category distribution:')
print(df['category'].value_counts())
print('\nGender distribution:')
print(df['gender'].value_counts(dropna=False))

## Step 3 — Handle Missing Values

Missing values (`NaN`) can distort analysis. We:
1. **Identify** how many are missing per column
2. **Fill** numeric columns with the median (robust to outliers)
3. **Fill** categorical columns with a placeholder string

### 3a. Count missing values per column

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing cells: {df.isnull().sum().sum()}')

### 3b. Missing values as percentages

In [ ]:
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print('Missing value % per column:')
print(missing_pct[missing_pct > 0])

### 3c. Fill missing values

| Column | Strategy | Reason |
|--------|----------|--------|
| `age` | Fill with **median** | Median is robust to extreme values |
| `rating` | Fill with **median** | Ratings are ordinal; median makes sense |
| `gender` | Fill with **`'Unknown'`** | Categorical — no meaningful numeric fill |

In [ ]:
# Pandas 3.x CoW-safe syntax (no inplace=True on chained operations)
df['age']    = df['age'].fillna(df['age'].median())
df['rating'] = df['rating'].fillna(df['rating'].median())
df['gender'] = df['gender'].fillna('Unknown')

print('After filling missing values:')
print(df.isnull().sum())
print(f'\nTotal missing cells remaining: {df.isnull().sum().sum()}')

## Step 4 — Basic Operations: Filter Rows & Select Columns

Pandas supports powerful indexing for slicing your data.

### 4a. Select specific columns

In [ ]:
# Select a subset of columns using a list
subset = df[['customer_id', 'product', 'category', 'price', 'quantity']]
print(f'Selected {subset.shape[1]} columns:')
subset.head()

### 4b. Filter rows — Electronics only

In [ ]:
# Single condition filter
electronics = df[df['category'] == 'Electronics']
print(f'Electronics purchases: {len(electronics)} rows')
electronics[['product', 'price', 'quantity']].head()

### 4c. Filter rows — High-value purchases (price > 500)

In [ ]:
high_value = df[df['price'] > 500]
print(f'High-value purchases (price > ₹500): {len(high_value)}')
high_value[['product', 'category', 'price']].head(8)

### 4d. Multi-condition filter — Female customers with rating ≥ 4

In [ ]:
# Use & for AND, | for OR; wrap each condition in ()
top_female = df[(df['gender'] == 'Female') & (df['rating'] >= 4)]
print(f'Female customers with rating ≥ 4: {len(top_female)}')
top_female[['customer_id', 'product', 'rating']].head()

### 4e. Sort data — Top 5 most expensive purchases

In [ ]:
top5 = df.sort_values('price', ascending=False).head(5)
top5[['product', 'category', 'price', 'quantity']]

## Step 5 — Remove Duplicates

Duplicate rows inflate counts and totals.  
We use `df.duplicated()` to detect them and `df.drop_duplicates()` to remove them.

In [ ]:
print(f'Rows BEFORE removing duplicates : {df.shape[0]}')
print(f'Duplicate rows detected         : {df.duplicated().sum()}')

# Drop duplicates and reset the row index
df = df.drop_duplicates().reset_index(drop=True)

print(f'Rows AFTER  removing duplicates : {df.shape[0]}')

## Step 6 — Create a Derived Column: `total_amount`

Derived columns are calculated from existing data.  

**Formula:** `total_amount = price × quantity`

In [ ]:
# Create the derived column
df['total_amount'] = df['price'] * df['quantity']

print('✅ New column "total_amount" added')
df[['product', 'price', 'quantity', 'total_amount']].head(10)

### 6b. Analysis using the new column

In [ ]:
print('total_amount statistics:')
print(df['total_amount'].describe().round(2))

print('\n📊 Total revenue by category:')
rev_by_cat = df.groupby('category')['total_amount'].sum().sort_values(ascending=False)
print(rev_by_cat.round(2))

print(f'\n💰 Grand total revenue: ₹{df["total_amount"].sum():,.2f}')

## Step 7 — Save the Cleaned Dataset

We export the cleaned and enriched DataFrame to a new CSV file using `df.to_csv()`.  
Setting `index=False` prevents Pandas from writing the row numbers as a column.

In [ ]:
output_path = 'shopping_cleaned.csv'
df.to_csv(output_path, index=False)

# Verify the saved file
df_verify = pd.read_csv(output_path)
print(f'✅ Cleaned dataset saved to: {output_path}')
print(f'   Final shape  : {df_verify.shape[0]} rows × {df_verify.shape[1]} columns')
print(f'   Columns      : {df_verify.columns.tolist()}')
print(f'   Missing values: {df_verify.isnull().sum().sum()}')

---
## 📋 Project Summary

| Step | Task | Result |
|------|------|--------|
| 1 | Load CSV into Pandas DataFrame | 204 rows × 9 columns loaded |
| 2 | Explore data (head/tail, shape, dtypes, describe) | 5 categories, 3 genders, prices ₹5–₹2000 |
| 3 | Handle missing values | `age` & `rating` → median; `gender` → 'Unknown' |
| 4 | Filter rows & select columns | Filtered by category, price, gender+rating |
| 5 | Remove duplicate rows | 4 duplicates removed → 200 rows |
| 6 | Create `total_amount = price × quantity` | Grand total revenue: ₹368,042 |
| 7 | Save cleaned dataset | `shopping_cleaned.csv` — 200 rows × 10 columns |

### Key Takeaways
- **Pandas DataFrames** are the backbone of Python data analysis
- Always **inspect your data** before cleaning (shape, dtypes, missing values)
- **Median** is preferred over mean for filling missing numeric values (outlier-robust)
- **Derived columns** add analytical value without changing raw data
- Always save cleaned data to a **new file** — never overwrite raw data